# Salary Linear Regression Notebook Training - KiloCode

Notebook nay la noi train model Linear Regression. File `modeling.salary_regression` chi la helper/backend de notebook goi lai, giup luu model va artifact cho Streamlit UI.

## Caveats

- Day la baseline hoc Linear Regression, khong phai ket luan luong thi truong.
- File salary chi gom job co numeric salary. TopDev hien khong co numeric salary nen khong nam trong model.
- Coefficient cua Linear Regression la tin hieu lien he trong model, khong phai bang chung nhan qua.
- Target dung log salary de giam skew va outlier influence.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "analysis" / "salary_analysis_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis/salary_analysis_clean.csv")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SALARY_PATH = REPO_ROOT / "data" / "analysis" / "salary_analysis_clean.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "modeling" / "salary_regression" / "kilocode"

print(f"Repo root: {REPO_ROOT}")
print(f"Salary input: {SALARY_PATH}")
print(f"Notebook train output dir: {OUTPUT_DIR}")

In [ ]:
from modeling.salary_regression import (
    ARTIFACT_FILENAMES,
    MODEL_FILENAME,
    prepare_salary_modeling_data,
    fit_salary_linear_regression,
    write_outputs,
)

## 1. Load Data

Cell nay kiem tra nhanh dung file va coverage truoc khi train model.

In [ ]:
salary = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")

print(f"Rows: {len(salary):,}")
print(f"Unique URLs: {salary['url'].nunique():,}")
display(salary[["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "salary_midpoint", "seniority", "experience_min"]].head(10))
display(pd.crosstab(salary["source"], salary["salary_currency"], dropna=False))

## 2. Prepare Modeling Data

Ham prepare lam cac viec chinh:

- Convert salary ve VND/thang.
- Tao target `log_salary_monthly_vnd`.
- Chuan hoa location va skills.
- Tao feature numeric, categorical va top skill flags.
- Chan leakage columns nhu `salary_raw`, `salary_midpoint`, `salary_currency`.

In [ ]:
modeling_data = prepare_salary_modeling_data(
    salary,
    top_skills=30,
    min_skill_count=5,
    usd_to_vnd=26_000,
)

print(f"Modeling rows: {len(modeling_data.frame):,}")
print(f"Feature count: {len(modeling_data.feature_columns):,}")
print(f"Top skills: {modeling_data.top_skills[:10]}")
display(modeling_data.audit.head(20))
display(modeling_data.frame[["source", "salary_raw", "salary_currency", "salary_period", "salary_period_clean", "salary_monthly_vnd_m", "location_norm", "skill_count", "log_salary_monthly_vnd"]].head(10))

## 3. Train Linear Regression

Model chi dung `sklearn.linear_model.LinearRegression`. Pipeline tu dong impute numeric/categorical, scale numeric, va one-hot encode categorical.

In [ ]:
result = fit_salary_linear_regression(
    modeling_data,
    test_size=0.2,
    random_state=42,
)

print(f"Train rows: {result.train_rows:,}")
print(f"Test rows: {result.test_rows:,}")
display(result.metrics)

## 4. Read Metrics

Cach doc nhanh:

- `mae_log` va `rmse_log`: loi tren log salary. Nho hon la tot hon.
- `r2_log`: ty le bien thien log salary model giai thich duoc. Gan 1 la tot, gan 0 la yeu.
- `mae_million_vnd`: trung binh sai bao nhieu trieu VND/thang sau khi convert nguoc bang exp.

## 5. Inspect Prediction Errors

Bang nay sap xep job sai nhieu nhat o test set. Day la noi tot de hoc model bi lech o dau: source nao, seniority nao, skill nao, salary outlier nao.

In [ ]:
display(result.predictions.head(20))

error_summary = result.predictions["abs_error_million_vnd"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
display(error_summary.to_frame(name="abs_error_million_vnd"))

## 6. Inspect Coefficients

Coefficient duong nghia la feature lam log salary du doan cao hon khi cac feature khac giu nguyen trong model. Coefficient am nghia la feature gan voi log salary du doan thap hon. Vi co one-hot va scaling, hay doc coefficient nhu tin hieu hoc tap, khong doc nhu nhan qua.

In [ ]:
display(result.coefficients.head(30))
display(result.coefficients.sort_values("coefficient", ascending=False).head(15))
display(result.coefficients.sort_values("coefficient", ascending=True).head(15))

## 7. Save Notebook-Trained Model For Streamlit

Cell nay la buoc luu model sau khi train trong notebook. Sau khi chay cell nay, Streamlit co the load `model.joblib` trong folder `kilocode`. CLI ben duoi chi la cach phu neu muon train lai tu terminal:

```bash
python -m modeling.salary_regression --input data/analysis/salary_analysis_clean.csv --output-dir data/modeling/salary_regression/kilocode
```

In [ ]:
write_outputs(result, OUTPUT_DIR)
artifact_rows = []
for filename in ARTIFACT_FILENAMES:
    path = OUTPUT_DIR / filename
    artifact_rows.append({
        "artifact": filename,
        "exists": path.exists(),
        "path": path,
        "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
    })

print(f"Notebook trained model file: {OUTPUT_DIR / MODEL_FILENAME}")
display(pd.DataFrame(artifact_rows))

## 8. Conclusion

Baseline nay du de hoc Linear Regression end-to-end: feature engineering, train/test, metrics, residuals va coefficients. Truoc khi dung cho ket luan thi truong, can them du lieu theo thoi gian, sua parser `salary_period`, va danh gia bias theo source.